In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q transformers torch librosa datasets scikit-learn tqdm

In [3]:
import os
import numpy as np
import torch
import librosa
from tqdm import tqdm

from transformers import Wav2Vec2Processor, Wav2Vec2Model

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

In [4]:
DATA_PATH = "/content/drive/MyDrive/SER_Project/"
RAVDESS_FOLDERS = [DATA_PATH + "Ravdess_Song/", DATA_PATH + "Ravdess_Speech/"]
TESS_FOLDER = DATA_PATH + "Tess/"

In [5]:
EMOTION_LABELS = [
    "neutral", "calm", "happy", "sad",
    "angry", "fearful", "surprise", "disgust"
]

LABEL2IDX = {l:i for i,l in enumerate(EMOTION_LABELS)}

In [6]:
emotion_map_ravdess = {
    "01": "neutral","02": "calm","03": "happy","04": "sad",
    "05": "angry","06": "fearful","07": "disgust","08": "surprise",
}

emotion_map_tess = {
    "neutral":"neutral","happy":"happy","sad":"sad",
    "angry":"angry","fear":"fearful","disgust":"disgust",
    "pleasant_surprise":"surprise",
}

In [7]:
def load_audio(file_path, sr=16000, max_len=5):
    audio, _ = librosa.load(file_path, sr=sr)
    max_samples = sr * max_len

    if len(audio) < max_samples:
        audio = np.pad(audio, (0, max_samples - len(audio)))
    else:
        audio = audio[:max_samples]

    return audio

In [8]:
def load_ravdess(folder_list):
    X, y = [], []

    for base in folder_list:
        for actor in os.listdir(base):
            actor_path = os.path.join(base, actor)

            if os.path.isdir(actor_path):
                for file in os.listdir(actor_path):
                    if file.endswith(".wav"):
                        parts = file.split("-")
                        if len(parts) < 3:
                            continue

                        label = emotion_map_ravdess.get(parts[2])

                        if label in LABEL2IDX:
                            X.append(load_audio(os.path.join(actor_path, file)))
                            y.append(LABEL2IDX[label])

    return X, y


def load_tess(folder):
    X, y = [], []

    for root,_,files in os.walk(folder):
        for file in files:
            if file.endswith(".wav"):
                name = file.split("_")[-1].replace(".wav","").lower()
                label = emotion_map_tess.get(name)

                if label in LABEL2IDX:
                    X.append(load_audio(os.path.join(root,file)))
                    y.append(LABEL2IDX[label])

    return X, y

In [9]:
X_r, y_r = load_ravdess(RAVDESS_FOLDERS)
X_t, y_t = load_tess(TESS_FOLDER)

X = np.array(X_r + X_t, dtype=np.float32)
y = np.array(y_r + y_t)

print("Total:", len(X))

Total: 4852


In [10]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.1, stratify=y, random_state=42
)

In [11]:
MODEL_ID = "jonatasgrosman/wav2vec2-large-xlsr-53-english"

processor = Wav2Vec2Processor.from_pretrained(MODEL_ID)
base_model = Wav2Vec2Model.from_pretrained(MODEL_ID)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/262 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.json:   0%|          | 0.00/300 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: jonatasgrosman/wav2vec2-large-xlsr-53-english
Key            | Status     |  | 
---------------+------------+--+-
lm_head.bias   | UNEXPECTED |  | 
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
class SERDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        audio = self.X[idx]
        label = self.y[idx]

        inputs = processor(
            audio,
            sampling_rate=16000,
            return_tensors="pt",
            padding=True
        )

        return inputs.input_values.squeeze(0), torch.tensor(label)

In [13]:
train_ds = SERDataset(X_train, y_train)
val_ds = SERDataset(X_val, y_val)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=4)

In [14]:
class SERModel(nn.Module):
    def __init__(self, base_model, num_labels):
        super().__init__()
        self.wav2vec = base_model

        self.classifier = nn.Sequential(
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_labels)
        )

    def forward(self, x):
        x = self.wav2vec(x).last_hidden_state
        x = x.mean(dim=1)
        return self.classifier(x)

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SERModel(base_model, len(EMOTION_LABELS)).to(device)

In [16]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
train_losses = []
val_losses = []
val_accuracies = []

EPOCHS = 10

for epoch in range(EPOCHS):

    # ================= TRAIN =================
    model.train()
    total_loss = 0

    for x, y in tqdm(train_loader):
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        out = model(x)

        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)
    train_losses.append(avg_train_loss)

    # ================= VALIDATION =================
    model.eval()
    val_loss = 0
    correct, total = 0, 0

    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)

            out = model(x)
            loss = criterion(out, y)
            val_loss += loss.item()

            preds = torch.argmax(out, dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    avg_val_loss = val_loss / len(val_loader)
    val_acc = correct / total

    val_losses.append(avg_val_loss)
    val_accuracies.append(val_acc)

    print(f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f}")

100%|██████████| 1092/1092 [16:32<00:00,  1.10it/s]


Epoch 1 | Train Loss: 0.9697 | Val Loss: 0.6109 | Val Acc: 0.7819


100%|██████████| 1092/1092 [16:53<00:00,  1.08it/s]


Epoch 2 | Train Loss: 0.4209 | Val Loss: 0.2818 | Val Acc: 0.8889


100%|██████████| 1092/1092 [16:54<00:00,  1.08it/s]


...

100%|██████████| 1092/1092 [16:53<00:00,  1.08it/s]


Epoch 4 | Train Loss: 0.2071 | Val Loss: 0.2098 | Val Acc: 0.9177


100%|██████████| 1092/1092 [16:53<00:00,  1.08it/s]


Epoch 5 | Train Loss: 0.1706 | Val Loss: 0.1583 | Val Acc: 0.9486


 82%|████████▏ | 891/1092 [13:46<03:09,  1.06it/s]

In [ ]:
SAVE_PATH = "/content/drive/MyDrive/SER_Project/ser_model.pth"

torch.save({
    "model_state": model.state_dict(),
    "labels": EMOTION_LABELS,
    "model_id": MODEL_ID
}, SAVE_PATH)

print("✅ Model saved at:", SAVE_PATH)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# ================= PREDICTIONS =================
model.eval()

y_true, y_pred = [], []

with torch.no_grad():
    for x, y in val_loader:
        x = x.to(device)

        outputs = model(x)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()

        y_true.extend(y.numpy())
        y_pred.extend(preds)

# ================= REPORT =================
print("\n===== CLASSIFICATION REPORT =====\n")
print(classification_report(y_true, y_pred, target_names=EMOTION_LABELS))

# ================= CONFUSION MATRIX =================
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=EMOTION_LABELS,
            yticklabels=EMOTION_LABELS)

plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

# ================= LOSS CURVES =================
plt.figure(figsize=(8,5))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.title("Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid()
plt.show()

# ================= ACCURACY CURVE =================
plt.figure(figsize=(8,5))
plt.plot(val_accuracies, label="Validation Accuracy")
plt.title("Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid()
plt.show()

In [ ]:
# model.eval()

# correct, total = 0, 0

# with torch.no_grad():
#     for x, y in val_loader:
#         x, y = x.to(device), y.to(device)

#         preds = model(x)
#         preds = torch.argmax(preds, dim=1)

#         correct += (preds == y).sum().item()
#         total += y.size(0)

# print("Accuracy:", correct / total)

In [ ]:
# y_true, y_pred = [], []

# with torch.no_grad():
#     for x, y in val_loader:
#         x = x.to(device)

#         out = model(x)
#         pred = torch.argmax(out, dim=1).cpu().numpy()

#         y_true.extend(y.numpy())
#         y_pred.extend(pred)

# print(classification_report(y_true, y_pred))

In [ ]:
# import matplotlib.pyplot as plt
# import seaborn as sns
# from sklearn.metrics import confusion_matrix, classification_report

# model.eval()

# y_true, y_pred = [], []

# with torch.no_grad():
#     for x, y in val_loader:
#         x = x.to(device)

#         outputs = model(x)
#         preds = torch.argmax(outputs, dim=1).cpu().numpy()

#         y_true.extend(y.numpy())
#         y_pred.extend(preds)

# # ================= REPORT =================
# print("\n===== CLASSIFICATION REPORT =====\n")
# print(classification_report(y_true, y_pred, target_names=EMOTION_LABELS))

# # ================= CONFUSION MATRIX =================
# cm = confusion_matrix(y_true, y_pred)

# plt.figure(figsize=(10,8))
# sns.heatmap(cm, annot=True, fmt="d",
#             xticklabels=EMOTION_LABELS,
#             yticklabels=EMOTION_LABELS)

# plt.title("Confusion Matrix")
# plt.xlabel("Predicted")
# plt.ylabel("True")
# plt.show()